In [ ]:
from groq import Groq
import re
import json
api = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
model = "openai/gpt-oss-20b"
groq = Groq(api_key=api)

In [2]:
def database(name: str):
    """Search the database by name and return the person's info."""
    data = {
        "yousef": {"name": "yousef", "age": 25, "major": "ai", "address": "cairo"},
        "ali": {"name": "ali", "age": 30, "major": "frontend", "address": "cairo"},
        "sama": {"name": "sama", "age": 27, "major": "data analysis", "address": "cairo"},
    }
    if name not in data:
        raise KeyError(f"'{name}' not found in database")
    return data[name]
 
 
def write_file(content):
    """Write data to a text file (final.txt)."""
    if isinstance(content, dict):
        content = "\n".join(f"{k}: {v}" for k, v in content.items())
    with open("final.txt", mode="w", encoding="utf-8") as file:
        file.write(str(content))
    return "final.txt written successfully"
 
 
TOOLS = {"database": database, "write_file": write_file}

In [4]:
# ---------------------------------------------------------------
# Planner
# ---------------------------------------------------------------
PLANNER_PROMPT = """
You are an AI planning system.
 
Your job is to create an executable plan for the user's request.
 
Available tools:
 
1. database(name) -> searches the database by name (string) and returns a
   dictionary with the person's information (name, age, major, address).
 
2. write_file(content) -> writes the given content to a text file.
 
If a step needs the output of a previous step, reference it using the
placeholder "{{step_N}}" where N is the id of that step.
Example: {"content": "{{step_1}}"}
Never invent or guess data that comes from a tool; always use a placeholder.
 
Return ONLY valid JSON.
 
Format:
 
{
    "steps": [
        {
            "id": 1,
            "task": "...",
            "tool": "...",
            "arguments": {}
        }
    ]
}
 
Do not execute any tool.
Only create the plan.
"""
 
 
def parse_json(text: str):
    """Parse JSON even if the model wrapped it in ```json fences."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)
 
 
def create_plan(request: str):
    response = groq.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": request},
        ],
    )
    return parse_json(response.choices[0].message.content)
 

In [5]:

PLACEHOLDER = re.compile(r"\{\{step_(\d+)\}\}")
 
 
def resolve_arguments(arguments: dict, results: dict):
    """Replace {{step_N}} placeholders with the real result of step N."""
    resolved = {}
    for key, value in arguments.items():
        if isinstance(value, str):
            full = re.fullmatch(r"\{\{step_(\d+)\}\}", value.strip())
            if full:
                # the whole value is a placeholder -> keep the original object (dict, etc.)
                resolved[key] = results[int(full.group(1))]
            else:
                # placeholder inside a longer string -> substitute as text
                resolved[key] = PLACEHOLDER.sub(
                    lambda m: str(results[int(m.group(1))]), value
                )
        else:
            resolved[key] = value
    return resolved
 

def execute_plan(plan: dict):
    results = {}
 
    for step in plan["steps"]:
        step_id = step["id"]
        task = step["task"]
        tool = step["tool"]
        arguments = resolve_arguments(step["arguments"], results)
 
        print(f"[ID]   : {step_id}")
        print(f"[TASK] : {task}")
        print(f"[TOOL] : {tool}")
        print(f"[ARGS] : {arguments}")
 
        results[step_id] = TOOLS[tool](**arguments)
 
    return results
 

In [6]:
# final answer
def generate_final_answer(user_input, plan, results):
    prompt = f"""
Answer the user's request using the execution results.
 
User request:
{user_input}
 
Plan:
{json.dumps(plan, indent=2)}
 
Execution results:
{json.dumps(results, indent=2)}
 
Give a clear final answer. Only state what is actually in the execution results.
"""
    response = groq.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content
 

In [7]:
def planning_agent(question: str):
    print("=" * 60)
    print("Build Planning.")
    print(f"QUESTION: {question}")
    print("=" * 60)
 
    plan = create_plan(question)
    print(f"PLAN: {plan}")
 
    print("=" * 60)
    print("Execute Plan..")
    results = execute_plan(plan)
    print(f"RESULTS: {results}")
 
    print("=" * 60)
    print("Final Answer")
    return generate_final_answer(question, plan, results)
 


In [8]:
q = "search by name = 'yousef'  in database and create text file and write in his information"
print(planning_agent(q))

Build Planning.
QUESTION: search by name = 'yousef'  in database and create text file and write in his information
PLAN: {'steps': [{'id': 1, 'task': "Search the database for a person named 'yousef'", 'tool': 'database', 'arguments': {'name': 'yousef'}}, {'id': 2, 'task': 'Write the retrieved information to a text file', 'tool': 'write_file', 'arguments': {'content': '{{step_1}}'}}]}
Execute Plan..
[ID]   : 1
[TASK] : Search the database for a person named 'yousef'
[TOOL] : database
[ARGS] : {'name': 'yousef'}
[ID]   : 2
[TASK] : Write the retrieved information to a text file
[TOOL] : write_file
[ARGS] : {'content': {'name': 'yousef', 'age': 25, 'major': 'ai', 'address': 'cairo'}}
RESULTS: {1: {'name': 'yousef', 'age': 25, 'major': 'ai', 'address': 'cairo'}, 2: 'final.txt written successfully'}
Final Answer
The database returned the following record:

- **Name:** yousef  
- **Age:** 25  
- **Major:** ai  
- **Address:** cairo  

The information was written to a text file named **final.